In [ ]:
import pandas as pd
import numpy as np

# Import data

In [ ]:
path1 = "/mnt/c/users/helen/Desktop/190226/WT/foci2/aggregated_data.csv"
path2 = "/mnt/c/users/helen/Desktop/190226/MGS3/foci2/aggregated_data.csv"

In [ ]:
df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)

In [ ]:
df1["sample"] = "WT"
df2["sample"] = "MGS3"

combined_df = pd.concat(
    [df1, df2],
    ignore_index=True
)

In [ ]:
# Clean data
combined_df.columns = combined_df.columns.str.strip().str.lower()
combined_df.iloc[:, 1:10] = combined_df.iloc[:, 1:10].apply(pd.to_numeric, errors="coerce")
df = combined_df.dropna()

print("Number of cells is:", df.shape[0])

# UMAP

In [ ]:
import umap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

features = [
    "n_foci",
    "area",
    "equivalent_diameter_area",
    "eccentricity",
    "perimeter",
    "solidity",
    "mean_intensity",
    "max_intensity",
]

# Select features
X = df[features]

# Standardize
X_scaled = StandardScaler().fit_transform(X)

# UMAP
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)

X_umap = reducer.fit_transform(X_scaled)

# Create dataframe
umap_df = df.copy()

umap_df["UMAP1"] = X_umap[:, 0]
umap_df["UMAP2"] = X_umap[:, 1]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse
from scipy.stats import chi2

def confidence_ellipse(x, y, ax, confidence=0.95, **kwargs):
    x = np.asarray(x)
    y = np.asarray(y)

    # Calculate covariance matrix
    cov = np.cov(x, y)

    # Eigenvalues and eigenvectors
    eigenvalues, eigenvectors = np.linalg.eigh(cov)

    # Sort in descending order
    order = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[order]
    eigenvectors = eigenvectors[:, order]

    # Ellipse rotation
    angle = np.degrees(
        np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])
    )

    # 95% confidence scaling
    scale = np.sqrt(chi2.ppf(confidence, df=2))

    width, height = 2 * scale * np.sqrt(eigenvalues)

    ellipse = Ellipse(
        xy=(np.mean(x), np.mean(y)),
        width=width,
        height=height,
        angle=angle,
        fill=False,
        linewidth=2,
        **kwargs
    )

    ax.add_patch(ellipse)


# Plot
fig, ax = plt.subplots(figsize=(9, 7))

palette = dict(zip(
    umap_df["sample"].unique(),
    sns.color_palette("Set2", umap_df["sample"].nunique())
))

sns.scatterplot(
    data=umap_df,
    x="UMAP1",
    y="UMAP2",
    hue="sample",
    palette=palette,
    s=80,
    alpha=0.7,
    edgecolor="white",
    ax=ax
)

# Add 95% ellipses
for group, data in umap_df.groupby("sample"):

    confidence_ellipse(
        data["UMAP1"],
        data["UMAP2"],
        ax,
        confidence=0.95,
        edgecolor=palette[group]
    )

    # Mark centroid
    ax.scatter(
        data["UMAP1"].mean(),
        data["UMAP2"].mean(),
        marker="X",
        s=180,
        color=palette[group],
        edgecolor="black",
        linewidth=1.2,
        zorder=5
    )

ax.set_xlabel("UMAP 1", fontsize=13)
ax.set_ylabel("UMAP 2", fontsize=13)
ax.set_title("UMAP — WT vs Mutant (95% Ellipses)", fontsize=15)

ax.legend(title="Sample", frameon=False)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.manifold import trustworthiness

score = trustworthiness(
    X_scaled,
    X_umap,
    n_neighbors=10
)

print(f"UMAP trustworthiness: {score:.4f}")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform
from skbio import DistanceMatrix
from skbio.stats.distance import permanova, permdisp

# Select features and groups
data = df[features + ["sample"]].dropna().copy()

X = data[features]
groups = data["sample"].to_numpy()

# Standardize features
X_scaled = StandardScaler().fit_transform(X)

# Euclidean distance matrix
distances = squareform(
    pdist(X_scaled, metric="euclidean")
)

dm = DistanceMatrix(distances)

# PERMANOVA
permanova_result = permanova(
    dm,
    grouping=groups,
    permutations=9999,
    seed=42
)

# PERMDISP
permdisp_result = permdisp(
    dm,
    grouping=groups,
    test="centroid",
    permutations=9999,
    seed=42
)

# Calculate R²
F = permanova_result["test statistic"]
n = len(groups)
k = len(np.unique(groups))

R2 = (F * (k - 1)) / (F * (k - 1) + (n - k))

# Results dataframe
results_df = pd.DataFrame({
    "Test": ["PERMANOVA", "PERMDISP"],
    "Statistic": [
        permanova_result["test statistic"],
        permdisp_result["test statistic"]
    ],
    "p_value": [
        permanova_result["p-value"],
        permdisp_result["p-value"]
    ],
    "R2": [R2, np.nan],
    "n": [n, n],
    "groups": [k, k]
})

display(results_df.round(4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, n in zip(axes.flat, [5, 10, 15, 20, 30, 50]):

    reducer = umap.UMAP(
        n_neighbors=n,
        min_dist=0.1,
        random_state=42
    )

    embedding = reducer.fit_transform(X_scaled)

    sns.scatterplot(
        x=embedding[:, 0],
        y=embedding[:, 1],
        hue=groups,
        palette="Set2",
        s=45,
        alpha=0.8,
        legend=False,
        ax=ax
    )

    ax.set_title(f"n_neighbors = {n}")
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import ttest_ind
import pandas as pd
import numpy as np

results = []

for feature in features:
    wt = df.loc[df["sample"] == "WT", feature].dropna()
    mgs3 = df.loc[df["sample"] == "MGS3", feature].dropna()

    # Cohen's d
    pooled_sd = np.sqrt(
        ((len(wt)-1)*wt.var(ddof=1) +
         (len(mgs3)-1)*mgs3.var(ddof=1)) /
        (len(wt)+len(mgs3)-2)
    )

    d = (mgs3.mean() - wt.mean()) / pooled_sd if pooled_sd > 0 else np.nan

    results.append({
        "Feature": feature,
        "WT_mean": wt.mean(),
        "MGS3_mean": mgs3.mean(),
        "Cohens_d": d,
        "p_value": ttest_ind(wt, mgs3, equal_var=False).pvalue
    })

feature_results = pd.DataFrame(results)

display(
    feature_results.sort_values(
        "Cohens_d",
        key=abs,
        ascending=False
    )
)